# Manual Barcode Binding Lab

Notebook for implementing and validating manual barcode binding with conflict detection, confirmation-based rebind, and regression-safe behavior.

## 1. Load Fixtures and In-Memory Store

In [ ]:
from copy import deepcopy
from dataclasses import dataclass
from typing import Dict, List, Optional

StoreCache = Dict[str, Dict[str, dict]]

fixtures: StoreCache = {
    "001": {
        "4601234567890": {"codes": ["A101", "A102"], "source": "backend"},
        "4600000000007": {"codes": ["A777"], "source": "manual"},
        "4821111111111": {"codes": ["A200"], "source": "backend"},
    },
    "002": {
        "4601234567890": {"codes": ["B001"], "source": "backend"},
        "5902222222222": {"codes": ["B777"], "source": "manual"},
    },
}

cache_snapshots = [deepcopy(fixtures)]
fixtures

## 2. Implement Baseline `resolveBarcode` Behavior

Simulate current auto-resolve priority: local store cache, optional API hit, fallback by item codes.

In [ ]:
def normalize_code_list(record: dict) -> List[str]:
    if not record:
        return []
    if isinstance(record.get("codes"), list):
        return [str(x) for x in record["codes"]]
    if record.get("code"):
        return [str(record["code"])]
    return []


def resolve_barcode(barcode: str, item_codes: List[str], store_number: str, cache: StoreCache) -> dict:
    barcode = str(barcode).strip()
    store_number = str(store_number)
    if not barcode:
        return {"resolved": False, "codes": [], "source": "empty"}

    store_cache = cache.get(store_number, {})
    local = store_cache.get(barcode)
    if local:
        codes = normalize_code_list(local)
        if codes:
            return {"resolved": True, "codes": sorted(set(codes)), "source": local.get("source", "cache")}

    # Simulated fallback: match if scanned value equals a known item code in this recount.
    if barcode in set(map(str, item_codes)):
        return {"resolved": True, "codes": [barcode], "source": "fallback"}

    return {"resolved": False, "codes": [], "source": "not_found"}


resolve_barcode("4601234567890", ["A101", "A999"], "001", fixtures)

## 3. Add Manual Bind API Shape (`bindItemToBarcode`)

Request fields:
- `currentBarcode`
- `itemCode`
- `storeNumber`
- `confirmRebind`

Response statuses:
- `conflict`
- `bound`
- `noop`

In [ ]:
@dataclass
class BindRequest:
    currentBarcode: str
    itemCode: str
    storeNumber: str
    confirmRebind: bool = False


def bind_response(status: str, **kwargs) -> dict:
    payload = {"status": status}
    payload.update(kwargs)
    return payload

## 4. Detect Cross-Barcode Conflict for Selected `itemCode`

Find an existing barcode containing the selected item in the same `storeNumber` scope.

In [ ]:
def find_conflict_barcode(store_cache: Dict[str, dict], current_barcode: str, item_code: str) -> Optional[str]:
    item_code = str(item_code)
    for barcode, record in store_cache.items():
        if barcode == current_barcode:
            continue
        if item_code in normalize_code_list(record):
            return barcode
    return None


find_conflict_barcode(fixtures["001"], "4821111111111", "A101")

## 5. Handle User Decision: Cancel vs Confirm Rebind

If conflict exists and `confirmRebind` is false, return conflict payload and do not mutate state.
If `confirmRebind` is true, continue to atomic rebind.

## 6. Apply Atomic Rebind with De-duplication

Single transaction on an immutable copy:
- remove `itemCode` from old barcode
- add `itemCode` to current barcode
- dedupe resulting `codes` lists
- commit only if operation completes

In [ ]:
def atomic_bind_item_to_barcode(req: BindRequest, cache: StoreCache) -> tuple[dict, StoreCache]:
    store = str(req.storeNumber)
    current_barcode = str(req.currentBarcode).strip()
    item_code = str(req.itemCode).strip()

    if not current_barcode or not item_code:
        return bind_response("noop", reason="invalid_request"), cache

    if store not in cache:
        return bind_response("noop", reason="unknown_store"), cache

    next_cache = deepcopy(cache)
    store_cache = next_cache.setdefault(store, {})

    conflict = find_conflict_barcode(store_cache, current_barcode, item_code)
    if conflict and not req.confirmRebind:
        return bind_response(
            "conflict",
            currentBarcode=current_barcode,
            itemCode=item_code,
            conflictBarcode=conflict,
            message=f"Код {item_code} уже привязан к {conflict}. Подтвердите перепривязку на {current_barcode}."
        ), cache

    # Remove from all other barcodes in this store scope.
    for barcode, record in list(store_cache.items()):
        codes = normalize_code_list(record)
        cleaned = [c for c in codes if c != item_code or barcode == current_barcode]
        if barcode != current_barcode:
            cleaned = [c for c in cleaned if c != item_code]

        if cleaned:
            record["codes"] = sorted(set(cleaned))
            store_cache[barcode] = record
        else:
            store_cache.pop(barcode, None)

    current = store_cache.get(current_barcode, {"codes": [], "source": "manual"})
    merged = sorted(set(normalize_code_list(current) + [item_code]))
    current["codes"] = merged
    current["source"] = "manual"
    store_cache[current_barcode] = current

    return bind_response(
        "bound",
        currentBarcode=current_barcode,
        itemCode=item_code,
        movedFrom=conflict,
        codes=merged,
    ), next_cache


req = BindRequest(currentBarcode="4821111111111", itemCode="A101", storeNumber="001", confirmRebind=False)
atomic_bind_item_to_barcode(req, fixtures)[0]

## 7. Preserve Compatibility (`storeNumber`, cache, fallback)

Demonstrate unchanged behavior of `resolve_barcode` after manual bind operations and store scoping.

In [ ]:
base_cache = deepcopy(fixtures)

# Confirm rebind for store 001.
req_confirm = BindRequest(currentBarcode="4821111111111", itemCode="A101", storeNumber="001", confirmRebind=True)
res_bind, new_cache = atomic_bind_item_to_barcode(req_confirm, base_cache)

# Resolve still works in same store and unchanged in another store.
resolved_new = resolve_barcode("4821111111111", ["A101"], "001", new_cache)
resolved_old = resolve_barcode("4601234567890", ["A101"], "001", new_cache)
resolved_other_store = resolve_barcode("4601234567890", ["B001"], "002", new_cache)

res_bind, resolved_new, resolved_old, resolved_other_store

## 8. Write Unit Tests for Conflict and Regression Scenarios

Runnable tests:
- conflict payload contains conflicting barcode
- cancel leaves cache unchanged
- confirm moves code correctly
- repeated bind creates no duplicates
- baseline resolve behavior remains valid

In [ ]:
def run_tests():
    local = deepcopy(fixtures)

    # 1) conflict payload
    req = BindRequest(currentBarcode="4821111111111", itemCode="A101", storeNumber="001", confirmRebind=False)
    res, after = atomic_bind_item_to_barcode(req, local)
    assert res["status"] == "conflict"
    assert res["conflictBarcode"] == "4601234567890"
    assert after == local

    # 2) cancel leaves links unchanged
    before = deepcopy(local)
    res_cancel, after_cancel = atomic_bind_item_to_barcode(req, local)
    assert res_cancel["status"] == "conflict"
    assert after_cancel == before

    # 3) confirm moves link
    req_confirm = BindRequest(currentBarcode="4821111111111", itemCode="A101", storeNumber="001", confirmRebind=True)
    res_ok, moved = atomic_bind_item_to_barcode(req_confirm, local)
    assert res_ok["status"] == "bound"
    assert "A101" in normalize_code_list(moved["001"]["4821111111111"])
    assert "A101" not in normalize_code_list(moved["001"]["4601234567890"])

    # 4) repeated bind no duplicates
    res_again, moved_again = atomic_bind_item_to_barcode(req_confirm, moved)
    codes = normalize_code_list(moved_again["001"]["4821111111111"])
    assert codes.count("A101") == 1
    assert res_again["status"] == "bound"

    # 5) resolve regressions
    out_cache = resolve_barcode("4601234567890", ["A101", "A102"], "001", moved_again)
    out_fallback = resolve_barcode("A404", ["A404", "A999"], "001", moved_again)
    assert out_cache["resolved"] is True
    assert out_fallback == {"resolved": True, "codes": ["A404"], "source": "fallback"}

    print("All tests passed")


run_tests()